# Sepsis Early Warning — Patient Exploration

This notebook contains the first exploratory steps of the project. The aim is to understand the dataset before building any machine-learning model.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

## 1. Load one patient

Each PhysioNet `.psv` file represents one patient. Here we load patient `p000034` directly from the public dataset.

In [ ]:
base_url = "https://physionet.org/files/challenge-2019/1.0.0/training/training_setA/"

patient_id = "p000034"
df = pd.read_csv(base_url + patient_id + ".psv", sep="|")

df

In [ ]:
df[[
    "ICULOS",
    "HR",
    "O2Sat",
    "Temp",
    "SBP",
    "MAP",
    "Resp",
    "SepsisLabel"
]]

## 2. Missing data

`NaN` means that a value was not recorded for that variable at that observation. Missing clinical measurements must not automatically be treated as zero.

In [ ]:
# Number of missing values for each variable
df.isna().sum()

In [ ]:
# Percentage of missing values, ordered from highest to lowest
missing_percentage = df.isna().mean() * 100
missing_percentage.sort_values(ascending=False)

## 3. Heart rate over time

In [ ]:
plt.plot(df["ICULOS"], df["HR"], marker="o")
plt.xlabel("Hours in ICU")
plt.ylabel("Heart Rate (bpm)")
plt.title("Heart Rate over ICU stay")
plt.show()

## 4. First positive sepsis warning label

We identify the first ICU time at which `SepsisLabel` becomes 1 and visualize it together with heart rate.

In [ ]:
sepsis_start = df.loc[
    df["SepsisLabel"] == 1,
    "ICULOS"
].min()

print(sepsis_start)

In [ ]:
plt.plot(df["ICULOS"], df["HR"], marker="o")

plt.axvline(
    x=sepsis_start,
    linestyle="--",
    label="Sepsis warning label"
)

plt.xlabel("Hours in ICU")
plt.ylabel("Heart Rate (bpm)")
plt.title("Heart Rate and sepsis warning label")
plt.legend()
plt.show()

## 5. Summarize one patient

We now inspect another patient and convert the full time series into a small descriptive summary. This is useful for learning, but later we will avoid using future measurements when constructing predictive features.

In [ ]:
patient_id = "p000001"
url = base_url + patient_id + ".psv"
patient = pd.read_csv(url, sep="|")

patient.head()

In [ ]:
# Whether this patient ever receives a positive SepsisLabel
patient["SepsisLabel"].max()

In [ ]:
# Number of observations/rows for this patient — NOT the number of patients in the dataset
len(patient)

In [ ]:
mean_hr = patient["HR"].mean()
min_o2 = patient["O2Sat"].min()

print("Mean HR:", mean_hr)
print("Minimum O2 saturation:", min_o2)

In [ ]:
summary = {
    "patient": patient_id,
    "hours_in_icu": patient["ICULOS"].max(),
    "septic": patient["SepsisLabel"].max(),
    "mean_HR": patient["HR"].mean(),
    "min_O2Sat": patient["O2Sat"].min(),
    "mean_Resp": patient["Resp"].mean(),
    "mean_MAP": patient["MAP"].mean()
}

summary

## 6. Compare the first 20 patients

Here one row of the resulting DataFrame represents one patient. `septic = 0` means no positive sepsis label was observed in that patient's file; it does **not** mean that the patient was generally healthy.

In [ ]:
patients_summary = []

for i in range(1, 21):
    patient_id = f"p{i:06d}"
    url = base_url + patient_id + ".psv"
    patient = pd.read_csv(url, sep="|")

    summary = {
        "patient": patient_id,
        "hours_in_icu": patient["ICULOS"].max(),
        "septic": patient["SepsisLabel"].max(),
        "mean_HR": patient["HR"].mean(),
        "min_O2Sat": patient["O2Sat"].min(),
        "mean_Resp": patient["Resp"].mean(),
        "mean_MAP": patient["MAP"].mean()
    }

    patients_summary.append(summary)

summary_df = pd.DataFrame(patients_summary)
summary_df

In [ ]:
summary_df["septic"].value_counts()

In [ ]:
summary_df.groupby("septic")[[
    "mean_HR",
    "min_O2Sat",
    "mean_Resp",
    "mean_MAP"
]].mean()

## 7. Avoiding data leakage

The summaries above use the entire patient stay. That is acceptable for descriptive exploration, but not for prediction: measurements occurring after the prediction time would leak future information into the model.

As a first learning exercise, we split the first six observations from everything that follows.

In [ ]:
patient_id = "p000001"
patient = pd.read_csv(base_url + patient_id + ".psv", sep="|")

first_6h = patient.head(6)
future = patient.iloc[6:]

display(first_6h[["ICULOS", "HR", "O2Sat", "MAP", "Resp", "SepsisLabel"]])
display(future[["ICULOS", "HR", "O2Sat", "MAP", "Resp", "SepsisLabel"]].head())

In [ ]:
early_sepsis = first_6h["SepsisLabel"].max()
future_sepsis = future["SepsisLabel"].max()

print("Positive label in first 6 observations:", early_sepsis)
print("Positive label later in the stay:", future_sepsis)

## 8. First simple features

Features are numerical summaries that can later be supplied to a predictive model. Here we calculate simple statistics using only the initial observation window.

In [ ]:
mean_hr_6h = first_6h["HR"].mean()
min_o2_6h = first_6h["O2Sat"].min()
mean_resp_6h = first_6h["Resp"].mean()
mean_map_6h = first_6h["MAP"].mean()

print("Mean HR:", mean_hr_6h)
print("Minimum O2Sat:", min_o2_6h)
print("Mean Resp:", mean_resp_6h)
print("Mean MAP:", mean_map_6h)

In [ ]:
hr_min_6h = first_6h["HR"].min()
hr_max_6h = first_6h["HR"].max()
hr_std_6h = first_6h["HR"].std()

print("HR minimum:", hr_min_6h)
print("HR maximum:", hr_max_6h)
print("HR standard deviation:", hr_std_6h)

### Missing-safe HR change

Using the first and last row directly can fail when one of them contains a missing HR value. We first remove missing HR observations and compute a change only if at least two measurements are available.

This is still a very crude trend feature; later we can replace it with a more robust slope based on all available time points.

In [ ]:
valid_hr = first_6h["HR"].dropna()

if len(valid_hr) >= 2:
    hr_start = valid_hr.iloc[0]
    hr_end = valid_hr.iloc[-1]
    hr_change = hr_end - hr_start
else:
    hr_start = float("nan")
    hr_end = float("nan")
    hr_change = float("nan")

print("First available HR:", hr_start)
print("Last available HR:", hr_end)
print("HR change:", hr_change)

## Current checkpoint

At this point the project can:
- inspect individual patients;
- quantify missing data;
- visualize heart rate and sepsis labels;
- summarize multiple patients;
- distinguish descriptive full-stay analysis from predictive analysis;
- isolate an initial observation window;
- compute basic features while accounting for missing HR measurements.

**Next:** define the prediction window more rigorously and wrap patient processing into a reusable Python function.